In [2]:
import pandas as pd
from pprint import pprint
import json
import re
from typing import Dict, List, Any

In [4]:
ds_workouts_101 = pd.read_json("Original datasets/workouts_101.json", lines=True)

In [5]:
samples = []
# pprint(ds_7dayWorkout_900['Context'][0]+ds_7dayWorkout_900['Response'][0])
for row in ds_workouts_101.index:
    samples.append({col: ds_workouts_101[col][row] for col in ds_workouts_101.columns})
pprint(len(samples))

101


In [7]:
def save_for_finetune(converted, filename="converted_dataset.jsonl"):
    """
    Save the converted dataset in JSONL format for LLM fine-tuning.

    Args:
        converted (list): List of schema dicts produced by preprocess_dataset.
        filename (str): Output file name.
    """
    with open(filename, "w", encoding="utf-8") as f:
        for sample in converted:
            # Each sample becomes one JSON line
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")

In [3]:
def load_mixed_json(file_path):
    objs = []
    decoder = json.JSONDecoder()

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    idx = 0
    while idx < len(content):
        # Skip whitespace
        while idx < len(content) and content[idx].isspace():
            idx += 1
        if idx >= len(content):
            break

        # Decode next JSON entity
        obj, end = decoder.raw_decode(content, idx)

        # Normalize: always append as list of dicts
        if isinstance(obj, dict):
            objs.append(obj)
        elif isinstance(obj, list):
            objs.extend(obj)
        else:
            raise ValueError(f"Unsupported JSON structure at index {idx}")

        idx = end

    return objs

In [14]:
def count_goals_and_tasks(samples):
    one_time_goals = 0
    non_repeatable_tasks = 0

    for sample in samples:
        # Count one-time goals
        if sample.get("goal_category") == "one_time":
            one_time_goals += 1

            # Count tasks with is_repeatable == False
            for task in sample.get("tasks", []):
                if task.get("is_repeatable") == False:
                    non_repeatable_tasks += 1

    return one_time_goals,non_repeatable_tasks
    

In [17]:
def convert_goals_and_tasks(samples):
    converted = []
    for sample in samples:
        # Update goal category
        sample["goal_category"] = "habit"

        # Update tasks
        for task in sample.get("tasks", []):
            task["is_repeatable"] = True

        converted.append(sample)
    return converted

In [5]:
samples=load_mixed_json("stage2/gemini_workouts_101.json")

In [18]:
samples=convert_goals_and_tasks(samples)

In [19]:
x,y=count_goals_and_tasks(samples)
print(x)
print (y)
# print(f"goals{x} \n tasks {y}")

0
0


In [21]:
save_for_finetune(samples, "final\workouts_101.jsonl")

In [9]:
print(samples[6
              ])

{'goal': 'Manage work-related stress using yoga and meditation for relaxation', 'goal_category': 'one_time', 'goal_type': 'fitness', 'time_horizon': 1, 'description': 'A calming 45-minute single-session routine combining deep breathing, gentle yoga, meditation, and mindful walking to promote mental clarity and relieve tension.', 'tasks': [{'index': 1, 'task': 'Deep Breathing', 'description': 'Breathwork to calm the mind: inhale for a count of 4, hold for 4, exhale for 4. Repeat continuously for 5 minutes. No structured rest.', 'is_repeatable': False, 'repeat_frequency': 1, 'gap_flag': False, 'estimated_duration_minutes': {'min': 5, 'max': 5}}, {'index': 2, 'task': 'Yoga Poses', 'description': "Gentle yoga flow: Child's Pose, Cat-Cow, Downward Dog, and Tree Pose. Hold each pose for 30 seconds to 1 minute. No structured rest.", 'is_repeatable': False, 'repeat_frequency': 1, 'gap_flag': False, 'estimated_duration_minutes': {'min': 15, 'max': 15}}, {'index': 3, 'task': 'Meditation', 'descr

{"text":"<s>[INST] Generate workout for this purpose: \"I want to get back in shape after being less active for a while. My goal is to achieve a well-rounded level of fitness, including strength, cardiovascular health, and flexibility\" [\/INST] The Total Fitness Revitalizer workout plan is designed to help you regain overall fitness by focusing on strength, cardiovascular health, and flexibility. This comprehensive program will guide you through a variety of exercises to enhance your strength, boost your cardiovascular endurance, and improve your flexibility, ensuring a well-rounded level of fitness.\\n\\n
**Exercise Recommendations**:\\n\\n1.
 **Warm-up (10 minutes)**: Begin with light cardio, such as brisk walking or jogging in place, to elevate your heart rate and prepare your body for the workout.\\n\\n2. 
 **Cardiovascular Conditioning (3 times a week, 20-30 minutes)**: Engage in activities like running, cycling, or swimming to improve your cardiovascular health. Alternate between high-intensity intervals and steady-state cardio for optimal results.\\n\\n3. 
 **Strength Training (3 times a week, 30 minutes)**: Focus on full-body strength exercises, including:\\n   a. Push-Ups: 3 sets of 12 reps.\\n   b. Bodyweight Squats: 3 sets of 15 reps.\\n   c. Dumbbell Rows: 3 sets of 12 reps per arm.\\n   d. Dumbbell Lunges: 3 sets of 12 reps per leg.\\n   e. Plank: Hold for 1 minute.\\n\\n4. 
 **Flexibility and Mobility (5 times a week, 15 minutes)**: Incorporate dynamic and static stretches to improve flexibility and enhance your range of motion. Focus on major muscle groups and joints.\\n\\n5. 
 **Balance and Stability (2 times a week, 15 minutes)**: Perform balance exercises like single-leg stands or stability ball exercises to enhance your core strength and stability.\\n\\n6. 
 **Cool Down and Relaxation (10 minutes)**: Finish each session with gentle stretching and deep breathing exercises to relax your muscles and calm your mind.\\n\\n
 **Total Exercise Time**: Approximately 60-75 minutes per session.\\n\\n
 **Rest**: Allow at least one day of complete rest per week to allow your body to recover. Listen to your body and adjust the intensity as needed.\\n\\nThe Total Fitness Revitalizer workout plan provides a balanced approach to fitness, ensuring you develop strength, cardiovascular endurance, flexibility, and stability. Consistency and dedication are key to achieving your well-rounded fitness goals. As you progress, consider increasing the intensity and duration of your workouts for continuous improvement. <\/s>"}


{"text":"<s>[INST] Generate workout for this purpose: \"I'm in my mid-30s and want to lose weight and tone my body for an upcoming beach vacation\" [\/INST] Are you ready to get in shape for your upcoming beach vacation? The Beach Vacation Body Sculptor workout is here to help you shed those extra pounds, tone your muscles, and get beach-ready! This workout plan combines various exercises targeting different muscle groups and includes both cardio and strength training.\\n\\n**Exercise Recommendations**:\\n1. **Warm-up (5 minutes)**: Start with a 5-minute light jog or jumping jacks to get your heart rate up.\\n\\n2. **Cardio Blast (10 minutes)**: Get your heart pumping with a combination of exercises such as jogging in place, high knees, and burpees. Perform each exercise for 1 minute, followed by a 30-second rest.\\n\\n3. **Upper Body Strength (15 minutes)**:\\n   a. Push-Ups: 3 sets of 10 reps with a 45-second rest in between.\\n   b. Dumbbell Bicep Curls: 3 sets of 12 reps with a 45-second rest.\\n   c. Tricep Dips: 3 sets of 10 reps with a 45-second rest.\\n   d. Bent-Over Rows: 3 sets of 12 reps with a 45-second rest.\\n\\n4. **Lower Body Sculpt (15 minutes)**:\\n   a. Squats: 3 sets of 15 reps with a 45-second rest.\\n   b. Lunges: 3 sets of 12 reps per leg with a 45-second rest.\\n   c. Leg Raises: 3 sets of 15 reps with a 45-second rest.\\n   d. Wall Sits: 3 sets of 30 seconds with a 45-second rest.\\n\\n5. **Core Crusher (10 minutes)**:\\n   a. Plank: Hold for 30 seconds, followed by a 15-second rest.\\n   b. Russian Twists: 3 sets of 12 reps per side with a 45-second rest.\\n   c. Bicycle Crunches: 3 sets of 15 reps per side with a 45-second rest.\\n   d. Leg Raises: 3 sets of 12 reps with a 45-second rest.\\n\\n6. **Cool Down and Stretching (5 minutes)**: Finish your workout with 5 minutes of stretching. Focus on your hamstrings, quads, calves, chest, back, and shoulders.\\n\\n**Total Workout Time**: Approximately 60 minutes\\n\\n**Rest**: 45 seconds between strength exercises, 30 seconds between cardio exercises, and 15 seconds during core exercises.\\n\\nThis workout plan is designed to help you lose weight, tone your body, and boost your fitness level. It's a well-rounded routine that combines strength training for muscle toning and cardio for fat burning. Remember to maintain proper form during exercises and stay hydrated. Enjoy your workout journey, and you'll be beach-ready in no time! <\/s>"}


You are a structured data transformation engine.

Convert the provided workout text into the exact JSON schema below.

Schema:
{
  "goal": string,
  "goal_category": "one_time" or "habit",
  "goal_type": string,
  "time_horizon": integer,
  "description": string,
  "tasks": [
    {
      "index": integer,
      "task": string,
      "description": string,
      "is_repeatable": boolean,
      "repeat_frequency": integer,
      "gap_flag": boolean,
      "estimated_duration_minutes": {
        "min": number,
        "max": number
      }
    }
  ]
}
Conversion Rules:

Goal Extraction

"goal" = summarize the user’s workout purpose clearly and concisely.

"goal_category" = "habit" if this workout is intended as an ongoing routine. Otherwise use "one_time".

"goal_type" = "fitness".

"time_horizon" = 30 if ongoing lifestyle improvement. Otherwise estimate logically.

"description" = short structured summary of the workout plan.

Task Breakdown

Break the workout into structured sections (Warm-up, Strength Training, Balance, Core, Cool Down).

Each section becomes ONE task.

Do NOT create a task per exercise unless explicitly structured that way.

Workout Focus

Emphasize:

Training type

Performance structure (sets, reps, hold times)

Rest intervals

Capture timing precisely.

If duration is given as a fixed time (e.g., 5 minutes), set:

"min": 5,
"max": 5

If duration is approximate (e.g., “20 minutes”), set a reasonable ±2 minute range.

Repeat Logic

"is_repeatable" = true for all workout tasks.

"repeat_frequency" = 1 (assume once per day unless otherwise specified).

gap_flag Rules (Important)

gap_flag = true ONLY if the task represents a complete rest day.

For all active workout sections → gap_flag = false.

Do NOT assign gap_flag = true to rest between sets.

Do NOT assign gap_flag = true to recovery instructions inside workouts.

Only full rest day = true.

Maintain:

Sequential indexing starting from 1.

Clean structured descriptions.

Valid JSON formatting.

No trailing commas.

Input:
{"text":"<s>[INST] Generate workout for this purpose: \"I have a sedentary job and need exercises to combat back pain and improve posture\" [\/INST] If you have a sedentary job and experience back pain or poor posture, this workout plan is tailored to help you alleviate discomfort and enhance your posture. It includes exercises that strengthen your core and back muscles while promoting flexibility.\\n\\n**Exercise Recommendations**:\\n\\n1. **Warm-up (5 minutes)**: Start with a 5-minute brisk walk to get your blood flowing.\\n\\n2. **Core Strengthening (3 days a week)**:\\n   - a. Planks: Hold for 30 seconds, followed by a 15-second rest.\\n   - b. Bird-Dogs: 3 sets of 10 reps per side with a 30-second rest.\\n   - c. Supermans: 3 sets of 12 reps with a 30-second rest.\\n\\n3. **Back and Shoulder Flexibility (2 days a week)**:\\n   - a. Cat-Cow Stretch: Perform for 2 minutes.\\n   - b. Child's Pose: Hold for 1 minute.\\n   - c. Shoulder Blade Squeeze: 3 sets of 12 reps with a 30-second rest.\\n\\n4. **Posture Awareness (Daily)**: Practice good posture while sitting at your desk. Sit up straight, keep your feet flat on the floor, and adjust your chair and computer to proper ergonomic positions.\\n\\n5. **Walking Breaks (Daily)**: Take a 5-minute walk break every hour to alleviate back strain.\\n\\n6. **Cool Down and Stretching (5 minutes)**: After your workout, spend 5 minutes stretching your back, shoulders, and chest muscles.\\n\\n7. **Rest Days (2 days a week)**: Allow your body to recover. No strenuous exercise on these days.\\n\\n**Total Workout Time**: 20-30 minutes for the exercise routine.\\n\\n**Rest**: 15 seconds during core exercises, 30 seconds during back and shoulder flexibility exercises.\\n\\nThis plan focuses on core strength and flexibility, which are crucial for improving posture and alleviating back pain. Consistency is key, so make an effort to maintain proper posture throughout the day and take regular walking breaks. <\/s>"}

Return only JSON.

You are a structured fitness dataset converter.

Transform the provided workout text into the exact JSON schema below.

Target Schema
{
  "goal": string,
  "goal_category": "one_time" or "habit",
  "goal_type": string,
  "time_horizon": integer,
  "description": string,
  "tasks": [
    {
      "index": integer,
      "task": string,
      "description": string,
      "is_repeatable": boolean,
      "repeat_frequency": integer,
      "gap_flag": boolean,
      "estimated_duration_minutes": {
        "min": number,
        "max": number
      }
    }
  ]
}
Transformation Rules
1. Goal Mapping

"goal" → summarize user’s fitness intention clearly.

"goal_category" →

"habit" if plan spans multiple days per week or implies routine.

"one_time" only if explicitly single session.

"goal_type" → "fitness"

"time_horizon" →

30 if ongoing weekly structure.

Otherwise estimate logically from context.

"description" → structured summary of the full program including frequency + emphasis.

2. Task Extraction Logic (Important)

Workout sections are NOT fixed. Detect structure dynamically.

Each of the following becomes ONE task:

Any top-level numbered section (Warm-up, Mobility, Aerobics, Stretching, Cool Down, etc.)

Any section that has its own frequency (e.g., “3 days a week”)

Rest Days MUST be a separate task

Do NOT create tasks per individual sub-exercise (a, b, c, d).
Instead:

Aggregate sub-exercises into the parent section task description.

Clearly describe sets, reps, rest, and structure inside the description.

3. Frequency Rules

Convert frequency into repeat_frequency:

“Daily” → 7

“3 days a week” → 3

“2 days a week” → 2

If unspecified → assume 1

Set:

"is_repeatable" = true for all workout-related tasks

"repeat_frequency" = weekly count (integer only)

4. gap_flag Rules (Strict)

gap_flag = true ONLY for complete rest days.

If the section is “Rest Days (2 days a week)” → create a task:

task: "Rest and Recovery"

repeat_frequency = 2

gap_flag = true

estimated_duration_minutes = { "min": 0, "max": 0 }

Never set gap_flag = true for:

Rest between sets

Cool down

Stretching

Recovery instructions inside workouts

Only full inactive days.

5. Duration Estimation

If duration is explicitly stated:

Fixed time → min = max

Range like “20–30 minutes” → use exact range

If duration must be estimated:

Calculate from sets, reps, hold times, and rest intervals logically.

Use realistic estimates.

Provide a small logical range if needed (±2–5 minutes).

If a section includes sub-parts (e.g., 20 min cardio + 10 min stretching):

Sum them.

6. Description Focus (Critical)

For each task description:

Emphasize:

Sets

Reps

Hold durations

Rest intervals

Performance structure

Keep it structured and compact.

No fluff.

7. Indexing

Start at 1.

Sequential order.

No missing numbers.

Input
{"text":"<s>[INST] Generate workout for this purpose: \"I have a sedentary job and need exercises to combat back pain and improve posture\" [\/INST] If you have a sedentary job and experience back pain or poor posture, this workout plan is tailored to help you alleviate discomfort and enhance your posture. It includes exercises that strengthen your core and back muscles while promoting flexibility.\\n\\n**Exercise Recommendations**:\\n\\n1. **Warm-up (5 minutes)**: Start with a 5-minute brisk walk to get your blood flowing.\\n\\n2. **Core Strengthening (3 days a week)**:\\n   - a. Planks: Hold for 30 seconds, followed by a 15-second rest.\\n   - b. Bird-Dogs: 3 sets of 10 reps per side with a 30-second rest.\\n   - c. Supermans: 3 sets of 12 reps with a 30-second rest.\\n\\n3. **Back and Shoulder Flexibility (2 days a week)**:\\n   - a. Cat-Cow Stretch: Perform for 2 minutes.\\n   - b. Child's Pose: Hold for 1 minute.\\n   - c. Shoulder Blade Squeeze: 3 sets of 12 reps with a 30-second rest.\\n\\n4. **Posture Awareness (Daily)**: Practice good posture while sitting at your desk. Sit up straight, keep your feet flat on the floor, and adjust your chair and computer to proper ergonomic positions.\\n\\n5. **Walking Breaks (Daily)**: Take a 5-minute walk break every hour to alleviate back strain.\\n\\n6. **Cool Down and Stretching (5 minutes)**: After your workout, spend 5 minutes stretching your back, shoulders, and chest muscles.\\n\\n7. **Rest Days (2 days a week)**: Allow your body to recover. No strenuous exercise on these days.\\n\\n**Total Workout Time**: 20-30 minutes for the exercise routine.\\n\\n**Rest**: 15 seconds during core exercises, 30 seconds during back and shoulder flexibility exercises.\\n\\nThis plan focuses on core strength and flexibility, which are crucial for improving posture and alleviating back pain. Consistency is key, so make an effort to maintain proper posture throughout the day and take regular walking breaks. <\/s>"}

**************ok 

You are a structured fitness dataset transformer.

Convert the provided workout text into the exact JSON schema below.

Target Schema
{
  "goal": string,
  "goal_category": "one_time" or "habit",
  "goal_type": string,
  "time_horizon": integer,
  "description": string,
  "tasks": [
    {
      "index": integer,
      "task": string,
      "description": string,
      "is_repeatable": boolean,
      "repeat_frequency": integer,
      "gap_flag": boolean,
      "estimated_duration_minutes": {
        "min": number,
        "max": number
      }
    }
  ]
}
Core Rules
1. Goal Mapping

"goal" → concise summary of the user's purpose.

"goal_category":

"habit" if recurring plan.

"one_time" if single session only.

"goal_type" → "fitness"

"time_horizon" → 30 if recurring weekly structure, otherwise estimate logically.

"description" → structured summary including frequency and focus.

2. Task Extraction (Dynamic)

Detect sections dynamically.

Each top-level numbered section becomes ONE task.

Sub-exercises (a, b, c, etc.) are aggregated into the parent task description.

Do NOT create tasks per sub-exercise.

3. Rest Day Handling (Strict)

If text says:
“Rest Days (2 days a week)”

You must:

Create TWO separate tasks.

Each task:

task: "Rest Day"

description: "Full recovery day. No structured training."

is_repeatable: true

repeat_frequency: 7

gap_flag: true

estimated_duration_minutes: { "min": 0, "max": 0 }

Each rest day is an individual task entry.

Never combine rest days into one task.

Never assign gap_flag=true to anything except full rest days.

4. Repeat Frequency Definition (Changed Logic)

repeat_frequency = interval in days between repetitions.

Use this mapping:

Daily → 1

Every other day → 2

3 days per week → 2 (approximate interval)

2 days per week → 3–4 (choose closest integer, usually 3)

Weekly → 7

If unspecified but recurring → 1

Always output a single integer.

5. Duration Rules

If fixed time given → min = max

If range given → use exact range

If computed from sets/reps/rest → estimate realistically

If multi-part (e.g., 20 min cardio + 10 min stretch) → sum durations

If unclear → provide reasonable min/max range

6. Description Focus

Each task description must emphasize:

Sets

Reps

Hold durations

Rest intervals

Performance structure

Intensity style (low-impact, strength-based, mobility-focused, etc.)

No fluff.

7. Indexing

Start from 1

Sequential

No gaps

Valid JSON

No trailing commas

Input
{"text":"<s>[INST] Generate workout for this purpose: \"I have a sedentary job and need exercises to combat back pain and improve posture\" [\/INST] If you have a sedentary job and experience back pain or poor posture, this workout plan is tailored to help you alleviate discomfort and enhance your posture. It includes exercises that strengthen your core and back muscles while promoting flexibility.\\n\\n**Exercise Recommendations**:\\n\\n1. **Warm-up (5 minutes)**: Start with a 5-minute brisk walk to get your blood flowing.\\n\\n2. **Core Strengthening (3 days a week)**:\\n   - a. Planks: Hold for 30 seconds, followed by a 15-second rest.\\n   - b. Bird-Dogs: 3 sets of 10 reps per side with a 30-second rest.\\n   - c. Supermans: 3 sets of 12 reps with a 30-second rest.\\n\\n3. **Back and Shoulder Flexibility (2 days a week)**:\\n   - a. Cat-Cow Stretch: Perform for 2 minutes.\\n   - b. Child's Pose: Hold for 1 minute.\\n   - c. Shoulder Blade Squeeze: 3 sets of 12 reps with a 30-second rest.\\n\\n4. **Posture Awareness (Daily)**: Practice good posture while sitting at your desk. Sit up straight, keep your feet flat on the floor, and adjust your chair and computer to proper ergonomic positions.\\n\\n5. **Walking Breaks (Daily)**: Take a 5-minute walk break every hour to alleviate back strain.\\n\\n6. **Cool Down and Stretching (5 minutes)**: After your workout, spend 5 minutes stretching your back, shoulders, and chest muscles.\\n\\n7. **Rest Days (2 days a week)**: Allow your body to recover. No strenuous exercise on these days.\\n\\n**Total Workout Time**: 20-30 minutes for the exercise routine.\\n\\n**Rest**: 15 seconds during core exercises, 30 seconds during back and shoulder flexibility exercises.\\n\\nThis plan focuses on core strength and flexibility, which are crucial for improving posture and alleviating back pain. Consistency is key, so make an effort to maintain proper posture throughout the day and take regular walking breaks. <\/s>"}




target schema :{
"goal": string,
"goal_category": "one_time" or "habit",
"goal_type": string,
"time_horizon": integer (days),
"description": string,
,"tasks": [
{
"index": integer,
"task": string,
"description": string,
"is_repeatable": boolean,
"repeat_frequency": integer,
"gap_flag": boolean,
"estimated_duration_minutes": {
"min": number,
"max": number
}
}
]
}